# Практическое занятие 5. Сбор текстовых данных с веб-страниц с использованием технологии Web Scraping

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/p05.ipynb)

**Цели занятия:**

- Знать правовые и этические правила веб-скрейпинга: `robots.txt`, условия сайта, вежливые задержки
- Загружать страницы библиотекой `requests`, понимать коды ответа HTTP и роль кодировки
- Извлекать данные из HTML с помощью BeautifulSoup: `find`, `find_all`, `select`
- Обходить пагинацию и страницы статей, сохранять собранные тексты в DataFrame и CSV

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/OWNER/REPO.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Что такое веб-скрейпинг

**Web Scraping** (веб-скрейпинг) — автоматический сбор данных с веб-страниц: программа скачивает HTML-код страницы и извлекает из него нужные фрагменты — заголовки, даты, тексты. Для компьютерной лингвистики это основной способ собрать **корпус**: новости, отзывы, форумы на узбекском языке, которых нет в готовых наборах данных.

| Шаг | Инструмент | Результат |
|---|---|---|
| 1. Загрузка | `requests.get(url)` | HTML-текст страницы |
| 2. Разбор | `BeautifulSoup(html, "html.parser")` | дерево тегов |
| 3. Извлечение | `find`, `find_all`, `select` | заголовки, ссылки, тексты |
| 4. Обход | `urljoin`, цикл по ссылкам | все страницы сайта |
| 5. Сохранение | `pandas` | CSV / JSON |

## Этика и право

Скрейпинг законен не всегда. Перед сбором данных проверьте:

- **robots.txt** — файл в корне сайта с правилами для роботов. `Disallow: /admin/` — туда ходить нельзя, `Crawl-delay: 1` — пауза между запросами не меньше 1 секунды;
- **условия использования** сайта (Terms of Service): некоторые сайты прямо запрещают автоматический сбор;
- **авторское право**: тексты принадлежат авторам. Для исследования корпус можно анализировать, но публиковать его целиком без разрешения нельзя;
- **персональные данные** (Закон Республики Узбекистан «О персональных данных»): не собирайте телефоны, e-mail, адреса людей;
- **нагрузку на сервер**: делайте паузы (`time.sleep`), не запускайте сотни параллельных запросов; если у сайта есть **API** — используйте его.

**robots.txt учебного сайта**

```text
User-agent: *
Allow: /
Crawl-delay: 1
```

> **Внимание: Правило вежливого робота**  
> Один запрос в 1–2 секунды, понятный User-Agent, соблюдение robots.txt и только те страницы, которые действительно нужны для исследования.

## HTTP-запрос и коды ответа

```python
import requests

r = requests.get(SITE + "index.html")
r.status_code                 # 200 — страница получена
r.headers.get("Content-Type") # 'text/html' — кодировка не указана!
r.encoding = "utf-8"          # ОБЯЗАТЕЛЬНО до обращения к r.text
html = r.text
r.raise_for_status()          # исключение, если код 4xx/5xx
```

| Код | Значение | Что делать |
|---|---|---|
| 200 | OK | разбирать страницу |
| 301, 302 | перенаправление | `requests` переходит сам |
| 403 | доступ запрещён | не обходить защиту |
| 404 | страница не найдена | пропустить ссылку |
| 429 | слишком много запросов | увеличить паузу |
| 500, 503 | ошибка сервера | повторить позже |

> **Заметка: Почему важна кодировка**  
> Если сервер не сообщил `charset`, `requests` считает текст кодировкой ISO-8859-1. Тогда «So‘nggi» превращается в «Soâ€˜nggi», а все узбекские апострофы ‘ ’ — в мусор. Задавайте `r.encoding = "utf-8"` или декодируйте байты сами: `r.content.decode("utf-8")`.

## BeautifulSoup: поиск по дереву тегов

```python
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")
soup.title.get_text()                         # текст тега <title>
soup.find("h2", class_="news-title")          # ПЕРВЫЙ подходящий тег или None
soup.find_all("article", class_="news-card")  # СПИСОК всех подходящих тегов
soup.select("h2.news-title a")                # CSS-селектор → список
soup.select_one("div.pagination a.next")      # первый по селектору или None
a = soup.select_one("h2.news-title a")
a["href"], a.get("href"), a.get_text(strip=True)
```

| CSS-селектор | Что выбирает |
|---|---|
| `p` | все теги p |
| `.lead` | теги с классом lead |
| `h2.news-title` | h2 с классом news-title |
| `div.meta span` | span на любой глубине внутри div.meta |
| `div.article-body > p` | p — прямые потомки div.article-body |
| `time[datetime]` | time, у которых есть атрибут datetime |

Структура учебного сайта: карточка `article.news-card` → `h2.news-title > a[href]`, `.meta` с `span.category`, `time[datetime]`, `span.author`; пагинация `div.pagination > a.next`. На странице статьи: `h1.article-title`, `div.article-body > p`, `ul.tags > li`, `span.views` и **рекламный блок** `aside.ads`, который в корпус попадать не должен.

## Относительные ссылки и пагинация

Ссылки в карточках относительные: `article/navroz-bayrami.html`. Полный адрес строится функцией `urljoin` относительно адреса текущей страницы:

```python
from urllib.parse import urljoin
import time

url, pages = SITE + "index.html", []
while url:                                   # пока есть «следующая» страница
    soup = get_soup(url)
    pages.append(url)
    nxt = soup.select_one("div.pagination a.next")
    url = urljoin(url, nxt["href"]) if nxt else None
    time.sleep(1)                            # Crawl-delay
```

> **Совет: Паузы в тренажёре**  
> Учебный сайт работает локально, поэтому в заданиях хватит `time.sleep(0.05)`. На реальных сайтах берите значение из `Crawl-delay` (обычно 1–3 с).

## Сохранение результата

```python
df = pd.DataFrame(rows)                          # rows — список словарей
df.to_csv("news.csv", index=False, encoding="utf-8")
```

## Задание 1. robots.txt и первый запрос

Перед сбором данных прочитайте правила сайта и скачайте его главную страницу.

1. Текст файла `SITE + "robots.txt"` уже в переменной `robots`. Найдите в нём строку `Crawl-delay` и запишите значение **числом** в `crawl_delay`.
2. Скачайте главную страницу `SITE + "index.html"` в переменную `r`, код ответа запишите в `status`.
3. Укажите кодировку `utf-8` и сохраните HTML-код страницы в `html` (строка).

<details><summary>💡 Подсказка</summary>

Переберите строки `robots.splitlines()`; у строки, начинающейся с `Crawl-delay`, возьмите часть после двоеточия: `int(line.split(":")[1])`. Код ответа — `r.status_code`, кодировка — `r.encoding = "utf-8"`.

</details>

In [ ]:
import requests

print("Адрес сайта:", SITE)
robots = requests.get(SITE + "robots.txt").text
print(robots)

crawl_delay = ...   # число из строки "Crawl-delay: ..."

r = requests.get(...)
status = ...
# задайте кодировку до обращения к r.text
html = r.text

print("Код ответа:", status)
print(html[:300])

## Задание 2. Этика веб-скрейпинга

_Это задание выполняется в веб-тренажёре (вопрос)._

Вы собираете корпус узбекских новостей для курсовой работы. Какие действия **допустимы**? Выберите все верные варианты.

## Задание 3. Разбор HTML: find, find_all, select

Переменная `html` уже содержит главную страницу. Создайте `soup = BeautifulSoup(html, "html.parser")` и найдите:

- `page_title` — текст тега `<title>`;
- `cards` — список всех карточек `article.news-card` (через `find_all`);
- `titles` — список заголовков новостей: текст ссылок внутри `h2.news-title`, полученных CSS-селектором (`select`);
- `next_href` — значение атрибута `href` у ссылки «Keyingi →» (`a` с классом `next`).

<details><summary>💡 Подсказка</summary>

`soup.title.get_text()`; `soup.find_all("article", class_="news-card")`; `[a.get_text(strip=True) for a in soup.select("h2.news-title a")]`; `soup.select_one("a.next")["href"]`.

</details>

In [ ]:
# Подготовительный код задания
import requests
r = requests.get(SITE + "index.html")
r.encoding = "utf-8"
html = r.text

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

page_title = ...
cards = ...
titles = ...
next_href = ...

print(page_title)
print(len(cards), "карточек")
for t in titles:
    print("-", t)
print("Следующая страница:", next_href)

## Задание 4. Данные из карточек новостей

Функция `get_soup(url)` уже определена (скачивает страницу с кодировкой UTF-8 и возвращает `BeautifulSoup`).

Напишите функцию `parse_cards(soup, page_url)`, которая возвращает **список словарей** — по одному на карточку — с ключами:

| Ключ | Откуда взять |
|---|---|
| `title` | текст ссылки в `h2.news-title` |
| `url` | **полный** адрес статьи: `urljoin(page_url, href)` |
| `category` | текст `span.category` |
| `date` | атрибут `datetime` тега `time` (формат `2026-09-01`) |
| `author` | текст `span.author` |

Примените её к главной странице и сохраните результат в DataFrame `news`.

<details><summary>💡 Подсказка</summary>

Атрибут тега — как ключ словаря: `card.select_one("time")["datetime"]`. Полный адрес: `urljoin(page_url, a["href"])`.

</details>

In [ ]:
# Подготовительный код задания
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_soup(url):
    """Скачать страницу и вернуть объект BeautifulSoup."""
    r = requests.get(url)
    r.raise_for_status()
    r.encoding = "utf-8"
    return BeautifulSoup(r.text, "html.parser")

In [ ]:
page_url = SITE + "index.html"
soup = get_soup(page_url)

def parse_cards(soup, page_url):
    rows = []
    for card in soup.select("article.news-card"):
        a = card.select_one("h2.news-title a")
        rows.append({
            "title": a.get_text(strip=True),
            "url": ...,
            "category": ...,
            "date": ...,
            "author": ...,
        })
    return rows

news = pd.DataFrame(parse_cards(soup, page_url))
news

## Задание 5. Обход пагинации

Новости разбиты на 3 страницы. Функции `get_soup` и `parse_cards` уже определены.

Начните с `SITE + "index.html"` и в цикле `while` переходите по ссылке `div.pagination a.next`, пока она есть:

- `pages` — список адресов всех посещённых страниц (полные адреса, по порядку);
- `news` — DataFrame со всеми карточками со всех страниц;
- между запросами делайте вежливую паузу `time.sleep(0.05)`.

Выведите число новостей и распределение по категориям.

<details><summary>💡 Подсказка</summary>

`nxt = soup.select_one("div.pagination a.next")`, затем `url = urljoin(url, nxt["href"]) if nxt else None`. Не забудьте удалить `break`.

</details>

In [ ]:
# Подготовительный код задания
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_soup(url):
    """Скачать страницу и вернуть объект BeautifulSoup."""
    r = requests.get(url)
    r.raise_for_status()
    r.encoding = "utf-8"
    return BeautifulSoup(r.text, "html.parser")

def parse_cards(soup, page_url):
    """Список словарей по карточкам новостей одной страницы-списка."""
    rows = []
    for card in soup.select("article.news-card"):
        a = card.select_one("h2.news-title a")
        rows.append({
            "title": a.get_text(strip=True),
            "url": urljoin(page_url, a["href"]),
            "category": card.select_one("span.category").get_text(strip=True),
            "date": card.select_one("time")["datetime"],
            "author": card.select_one("span.author").get_text(strip=True),
        })
    return rows

In [ ]:
url = SITE + "index.html"
pages = []
rows = []

while url:
    soup = get_soup(url)
    pages.append(url)
    rows.extend(parse_cards(soup, url))
    nxt = ...          # ссылка a.next или None
    url = ...          # полный адрес следующей страницы или None
    time.sleep(0.05)
    break              # уберите эту строку, когда допишете цикл

news = pd.DataFrame(rows)
print(len(pages), "страницы,", len(news), "новостей")
news["category"].value_counts()

## Задание 6. Страница статьи: текст без рекламы

Разберите статью `SITE + "article/navroz-bayrami.html"`:

- `title` — заголовок `h1.article-title`;
- `paragraphs` — список текстов абзацев **только** из `div.article-body` (каждый через `get_text(strip=True)`);
- `body` — абзацы, соединённые через `"\n"`;
- `tags` — список тегов из `ul.tags li`;
- `views` — число просмотров (целое) из `span.views` («Ko‘rishlar: 4005» → `4005`).

Рекламный блок `aside.ads` и ссылка «Bosh sahifaga qaytish» в текст попасть не должны.

<details><summary>💡 Подсказка</summary>

Селектор `div.article-body p` выбирает только абзацы статьи. Число из строки: `int(re.search(r"\d+", text).group())`.

</details>

In [ ]:
# Подготовительный код задания
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_soup(url):
    """Скачать страницу и вернуть объект BeautifulSoup."""
    r = requests.get(url)
    r.raise_for_status()
    r.encoding = "utf-8"
    return BeautifulSoup(r.text, "html.parser")

In [ ]:
import re

soup = get_soup(SITE + "article/navroz-bayrami.html")

title = soup.select_one("h1.article-title").get_text(strip=True)
paragraphs = [p.get_text(strip=True) for p in soup.find_all("p")]   # все ли это абзацы статьи?
body = "\n".join(paragraphs)
tags = ...
views = ...

print(title, "|", views, "просмотров |", tags)
print(body)

## Задание 7. Сбор всех статей и сохранение в CSV

DataFrame `news` (16 строк: title, url, category, date, author) уже собран обходом пагинации. Посетите **каждую** статью и дополните таблицу:

- `text` — текст статьи: абзацы `div.article-body p`, соединённые `"\n"` (без рекламы);
- `views` — число просмотров (int);
- `n_words` — число слов в тексте: `len(text.split())`.

Итог сохраните в DataFrame `articles` и в файл `news_site.csv` (`index=False`, кодировка UTF-8). Не забудьте паузу между запросами (`time.sleep(0.05)`).

<details><summary>💡 Подсказка</summary>

Внутри цикла: `paras = [p.get_text(strip=True) for p in soup.select("div.article-body p")]`, `texts.append("\n".join(paras))`; просмотры — `int(re.search(r"\d+", soup.select_one("span.views").get_text()).group())`. Сохранение: `articles.to_csv("news_site.csv", index=False, encoding="utf-8")`.

</details>

In [ ]:
# Подготовительный код задания
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_soup(url):
    """Скачать страницу и вернуть объект BeautifulSoup."""
    r = requests.get(url)
    r.raise_for_status()
    r.encoding = "utf-8"
    return BeautifulSoup(r.text, "html.parser")

def parse_cards(soup, page_url):
    """Список словарей по карточкам новостей одной страницы-списка."""
    rows = []
    for card in soup.select("article.news-card"):
        a = card.select_one("h2.news-title a")
        rows.append({
            "title": a.get_text(strip=True),
            "url": urljoin(page_url, a["href"]),
            "category": card.select_one("span.category").get_text(strip=True),
            "date": card.select_one("time")["datetime"],
            "author": card.select_one("span.author").get_text(strip=True),
        })
    return rows

url, rows = SITE + "index.html", []
while url:
    soup = get_soup(url)
    rows.extend(parse_cards(soup, url))
    nxt = soup.select_one("div.pagination a.next")
    url = urljoin(url, nxt["href"]) if nxt else None
news = pd.DataFrame(rows)

In [ ]:
import re

texts, views = [], []
for url in news["url"]:
    soup = get_soup(url)
    # извлеките текст статьи и число просмотров
    ...
    time.sleep(0.05)

articles = news.copy()
# articles["text"] = ...
# articles["views"] = ...
# articles["n_words"] = ...
# сохраните в news_site.csv

articles.head()

## Самостоятельная работа

## Самостоятельная работа

1. **Эссе (1–2 страницы):** «Перспективы применения технологий Big Data в компьютерной лингвистике» — какие узбекские веб-ресурсы (новости, форумы, соцсети) можно использовать как источники корпусов, какие правовые ограничения действуют.
2. Выберите реальный узбекоязычный новостной сайт. Изучите его `robots.txt` и условия использования, опишите в таблице CSS-селекторы для заголовка, даты, категории и текста статьи.
3. Если правила сайта это разрешают, соберите 20–30 статей с паузой не меньше 2 секунд и сохраните их в CSV (url, title, date, text). Посчитайте среднюю длину статьи в словах.